# Coffee sales: Excel dashboard reproduced in pandas

Recreates the Excel dashboard's aggregations in pandas for an equivalent, code-reviewable pipeline. Source: the same three tables (`orders`, `customers`, `products`) as the original workbook. The original workbook is not attached to this repository, so the sample rows below are five illustrative rows only — clearly not real totals — used to demonstrate the join/aggregation structure, not to report actual sales figures.

In [ ]:
import pandas as pd

# Illustrative structure only — replace with pd.read_excel("coffee_sales.xlsx", sheet_name=...)
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "customer_id": [101, 102, 101, 103, 102],
    "product_id": [201, 202, 203, 201, 202],
    "order_date": pd.to_datetime(["2024-01-03", "2024-01-04", "2024-01-10", "2024-01-11", "2024-01-15"]),
    "quantity": [2, 1, 3, 1, 2],
})
customers = pd.DataFrame({
    "customer_id": [101, 102, 103],
    "country": ["Australia", "New Zealand", "Australia"],
    "loyalty_tier": ["Gold", "Silver", "Gold"],
})
products = pd.DataFrame({
    "product_id": [201, 202, 203],
    "roast": ["Light", "Medium", "Dark"],
    "pack_size": ["250g", "1kg", "250g"],
    "unit_price": [12.0, 38.0, 13.5],
})

## Data model

Recreate the lookups/relationships Excel's `VLOOKUP` and data model did — `orders` is the fact table, joined to the two dimension tables.

In [ ]:
sales = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
)
sales["line_total"] = sales["quantity"] * sales["unit_price"]
sales

## Dashboard views

Recreate each dashboard view as a `groupby`: sales over time, sales by country, top-5 customers by spend.

In [ ]:
total_sales_over_time = sales.groupby(sales["order_date"].dt.date)["line_total"].sum()
sales_by_country = sales.groupby("country")["line_total"].sum()
top_customers = sales.groupby("customer_id")["line_total"].sum().nlargest(5)

total_sales_over_time, sales_by_country, top_customers

## Slicers as parameterised filters

Recreate the 3 Excel slicers (loyalty tier, roast, pack size) as a function that re-renders the aggregations for any filter combination — the same interactivity Excel slicers give a business user, expressed as a reusable, testable function.

In [ ]:
def filtered_dashboard(loyalty_tier=None, roast=None, pack_size=None):
    view = sales
    if loyalty_tier:
        view = view[view["loyalty_tier"] == loyalty_tier]
    if roast:
        view = view[view["roast"] == roast]
    if pack_size:
        view = view[view["pack_size"] == pack_size]
    return {
        "total_sales_over_time": view.groupby(view["order_date"].dt.date)["line_total"].sum(),
        "sales_by_country": view.groupby("country")["line_total"].sum(),
        "top_customers": view.groupby("customer_id")["line_total"].sum().nlargest(5),
    }

filtered_dashboard(loyalty_tier="Gold")

## Excel ↔ Python equivalence

The same interactivity Excel slicers give a business user is expressed above as a reusable, testable function — the join/aggregation logic is identical, only the rendering layer differs between a PivotTable and a pandas `groupby`.